In [0]:
dbutils.library.restartPython()

In [0]:
# Cell 1: Check Endpoint Status and Wait for Ready
from databricks.sdk import WorkspaceClient
import time

w = WorkspaceClient()
ENDPOINT_NAME = "gdpr-agent-staging"

def wait_for_endpoint_ready(max_wait_minutes=15):
    """Wait for endpoint to be ready"""
    print(f"🔍 Checking endpoint status...")
    
    start_time = time.time()
    max_wait_seconds = max_wait_minutes * 60
    
    while time.time() - start_time < max_wait_seconds:
        try:
            endpoint = w.serving_endpoints.get(ENDPOINT_NAME)
            state = str(endpoint.state.ready) if endpoint.state else "UNKNOWN"
            config_update = str(endpoint.state.config_update) if endpoint.state else "UNKNOWN"
            
            elapsed_minutes = (time.time() - start_time) / 60
            print(f"   Status: {state} | Config: {config_update} ({elapsed_minutes:.1f}m elapsed)")
            
            # Check if ready
            if "READY" in state and "NOT_UPDATING" in config_update:
                print(f"\n✅ Endpoint is READY!")
                print(f"   Name: {endpoint.name}")
                print(f"   URL: {endpoint.url}")
                return endpoint
            
            # Check if failed
            if "FAIL" in state or "FAIL" in config_update:
                print(f"\n❌ Endpoint deployment failed!")
                return None
            
            # Wait before checking again
            time.sleep(30)
            
        except Exception as e:
            print(f"   Error checking status: {e}")
            time.sleep(30)
    
    print(f"\n⏱️ Timeout waiting for endpoint (waited {max_wait_minutes} minutes)")
    return None

endpoint = wait_for_endpoint_ready(max_wait_minutes=15)

if endpoint is None:
    raise Exception("Endpoint is not ready. Please check the deployment.")

In [0]:
# Cell 2: Define Test Queries
test_queries = [
    "What are the GDPR requirements for data deletion?",
    "What is the maximum fine for GDPR violations?",
    "How long do we have to report a data breach?",
    "What is the right to be forgotten under GDPR?",
    "What are the lawful bases for processing personal data under GDPR?"
]

print("📝 Test Queries:")
for i, q in enumerate(test_queries, 1):
    print(f"   {i}. {q}")

In [0]:
# Cell 3: Send Test Requests to Endpoint
import requests
import json
from datetime import datetime

workspace_url = spark.conf.get("spark.databricks.workspaceUrl")
token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()

endpoint_url = f"https://{workspace_url}/serving-endpoints/{ENDPOINT_NAME}/invocations"

print("="*80)
print("🚀 SENDING TEST REQUESTS")
print("="*80)

results = []

for i, question in enumerate(test_queries, 1):
    print(f"\n📤 Request {i}/{len(test_queries)}: {question[:60]}...")
    
    payload = {
        "dataframe_split": {
            "columns": ["question"],
            "data": [[question]]
        }
    }
    
    try:
        start_time = datetime.now()
        
        response = requests.post(
            endpoint_url,
            headers={
                "Authorization": f"Bearer {token}",
                "Content-Type": "application/json"
            },
            json=payload,
            timeout=60
        )
        
        latency = (datetime.now() - start_time).total_seconds()
        
        if response.status_code == 200:
            result = response.json()
            answer = result['predictions'][0]['answer']
            
            print(f"✅ Success ({latency:.2f}s)")
            print(f"   Answer: {answer[:150]}...")
            
            results.append({
                'question': question,
                'answer': answer,
                'latency': latency,
                'status': 'success'
            })
        else:
            print(f"❌ Error: {response.status_code}")
            print(f"   {response.text[:200]}")
            
            results.append({
                'question': question,
                'error': response.text,
                'latency': latency,
                'status': 'error'
            })
            
    except Exception as e:
        print(f"❌ Exception: {str(e)}")
        results.append({
            'question': question,
            'error': str(e),
            'status': 'exception'
        })

print("\n" + "="*80)
print(f"✅ Sent {len(test_queries)} requests")
print(f"   Success: {sum(1 for r in results if r['status'] == 'success')}")
print(f"   Failed: {sum(1 for r in results if r['status'] != 'success')}")
print("="*80)

In [0]:
# Cell: Comprehensive Log Check and Troubleshooting

import time
from datetime import datetime

LOG_TABLE = "main.default.gdpr_agent_inference_logs"

print("="*80)
print("🔍 COMPREHENSIVE LOG CHECK")
print("="*80)

# Step 1: Check if table exists
print("\n1️⃣ Checking if logs table exists...")
try:
    tables = spark.sql("SHOW TABLES IN main.default").collect()
    table_names = [t.tableName for t in tables]
    
    if 'gdpr_agent_inference_logs' in table_names:
        print(f"✅ Table {LOG_TABLE} exists!")
        
        # Get row count
        count = spark.sql(f"SELECT COUNT(*) as cnt FROM {LOG_TABLE}").first().cnt
        print(f"   Row count: {count}")
        
        if count > 0:
            # Show recent logs
            print("\n📋 Recent logs:")
            recent_logs = spark.sql(f"""
                SELECT 
                    timestamp,
                    request_id,
                    question,
                    status,
                    latency_ms,
                    SUBSTRING(answer, 1, 100) as answer_preview
                FROM {LOG_TABLE}
                ORDER BY timestamp DESC
                LIMIT 10
            """)
            display(recent_logs)
            
            # Show stats
            stats = spark.sql(f"""
                SELECT 
                    COUNT(*) as total,
                    COUNT(DISTINCT DATE(timestamp)) as unique_days,
                    SUM(CASE WHEN status = 'success' THEN 1 ELSE 0 END) as successful,
                    SUM(CASE WHEN status = 'error' THEN 1 ELSE 0 END) as errors,
                    AVG(latency_ms) as avg_latency_ms,
                    MIN(timestamp) as first_log,
                    MAX(timestamp) as last_log
                FROM {LOG_TABLE}
            """).first()
            
            print("\n📊 Log Statistics:")
            print(f"   Total Logs: {stats.total}")
            print(f"   Successful: {stats.successful}")
            print(f"   Errors: {stats.errors}")
            print(f"   Avg Latency: {stats.avg_latency_ms:.2f}ms")
            print(f"   First Log: {stats.first_log}")
            print(f"   Last Log: {stats.last_log}")
            print(f"   Time since last log: {datetime.now() - stats.last_log}")
        else:
            print("⚠️  Table exists but is empty (0 rows)")
    else:
        print(f"❌ Table {LOG_TABLE} does NOT exist")
        print(f"\n📋 Existing tables in main.default:")
        for tname in sorted(table_names):
            print(f"   - {tname}")
            
except Exception as e:
    print(f"❌ Error checking table: {e}")

# Step 2: Check deployed model version
print("\n2️⃣ Checking deployed model version...")
try:
    endpoint = w.serving_endpoints.get(ENDPOINT_NAME)
    if endpoint.config and endpoint.config.served_entities:
        for entity in endpoint.config.served_entities:
            print(f"   Model: {entity.entity_name}")
            print(f"   Version: {entity.entity_version}")
            print(f"   Name: {entity.name}")
except Exception as e:
    print(f"   Error: {e}")

# Step 3: Try to manually create the table (if it doesn't exist)
print("\n3️⃣ Attempting to create logs table manually...")
try:
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {LOG_TABLE} (
            timestamp TIMESTAMP,
            request_id STRING,
            question STRING,
            answer STRING,
            context STRING,
            latency_ms DOUBLE,
            status STRING,
            error_message STRING,
            date DATE
        )
        USING DELTA
        PARTITIONED BY (date)
    """)
    print(f"✅ Table created/verified: {LOG_TABLE}")
except Exception as e:
    print(f"❌ Error creating table: {e}")

# Step 4: Check if we can write to the table manually
print("\n4️⃣ Testing manual write to logs table...")
try:
    test_data = [(
        datetime.now(),
        "test-request-id",
        "Test question",
        "Test answer",
        "Test context",
        100.0,
        "test",
        None,
        datetime.now().date()
    )]
    
    test_df = spark.createDataFrame(
        test_data,
        ["timestamp", "request_id", "question", "answer", "context", "latency_ms", "status", "error_message", "date"]
    )
    
    test_df.write.mode("append").saveAsTable(LOG_TABLE)
    print(f"✅ Successfully wrote test data to {LOG_TABLE}")
    
    # Verify
    count_after = spark.sql(f"SELECT COUNT(*) as cnt FROM {LOG_TABLE}").first().cnt
    print(f"   Table now has {count_after} rows")
    
    # Clean up test data
    spark.sql(f"DELETE FROM {LOG_TABLE} WHERE status = 'test'")
    print(f"   ✅ Test data cleaned up")
    
except Exception as e:
    print(f"❌ Error writing test data: {e}")

# Step 5: Recommendations
print("\n" + "="*80)
print("💡 TROUBLESHOOTING RECOMMENDATIONS")
print("="*80)

if 'gdpr_agent_inference_logs' not in table_names or count == 0:
    print("\n⚠️  Logs are not being created. Possible issues:")
    print("   1. The deployed model may not have the logging code yet")
    print("      → Check the model version deployed vs. the version you registered")
    print("   2. PySpark may not be available in the serving environment")
    print("      → The _write_logs_async method requires Spark")
    print("   3. The logging code may be failing silently")
    print("      → Check model serving logs in the Databricks UI")
    print("\n✅ Next steps:")
    print("   1. Re-register the model with logging code")
    print("   2. Redeploy to the endpoint")
    print("   3. Make a test request")
    print("   4. Wait 1-2 minutes and check again")
else:
    print("\n✅ Logs are working correctly!")
    print(f"   Found {count} log entries in the table")

In [0]:
# Cell 4: Wait for Logs to Appear (they may take 1-2 minutes)
import time

print("⏳ Waiting 2 minutes for logs to be written to Delta table...")
time.sleep(120)
print("✅ Wait complete!")

In [0]:
# Cell 5: Check Inference Logs Table
LOG_TABLE = "main.default.gdpr_agent_inference_logs"

print("="*80)
print(f"📊 CHECKING INFERENCE LOGS: {LOG_TABLE}")
print("="*80)

# Check if table exists
try:
    tables = spark.sql(f"SHOW TABLES IN main.default LIKE 'gdpr_agent_inference_logs'").collect()
    
    if len(tables) == 0:
        print(f"⚠️  Table {LOG_TABLE} does not exist yet.")
        print("   This is expected on first deployment.")
        print("   The table will be created when the endpoint processes its first request.")
        print("\n   Run the requests again and wait 2-3 minutes.")
    else:
        print(f"✅ Table exists: {LOG_TABLE}\n")
        
        # Query recent logs
        logs_df = spark.sql(f"""
            SELECT 
                timestamp,
                request_id,
                question,
                answer,
                latency_ms,
                status,
                error_message
            FROM {LOG_TABLE}
            ORDER BY timestamp DESC
            LIMIT 10
        """)
        
        log_count = logs_df.count()
        
        if log_count > 0:
            print(f"✅ Found {log_count} log entries!\n")
            display(logs_df)
            
            # Summary stats
            stats = spark.sql(f"""
                SELECT 
                    COUNT(*) as total_requests,
                    SUM(CASE WHEN status = 'success' THEN 1 ELSE 0 END) as successful,
                    SUM(CASE WHEN status = 'error' THEN 1 ELSE 0 END) as failed,
                    AVG(latency_ms) as avg_latency_ms,
                    MIN(timestamp) as first_request,
                    MAX(timestamp) as last_request
                FROM {LOG_TABLE}
            """).first()
            
            print("\n📈 Summary Statistics:")
            print(f"   Total Requests: {stats.total_requests}")
            print(f"   Successful: {stats.successful}")
            print(f"   Failed: {stats.failed}")
            print(f"   Avg Latency: {stats.avg_latency_ms:.2f}ms")
            print(f"   First Request: {stats.first_request}")
            print(f"   Last Request: {stats.last_request}")
        else:
            print("⚠️  Table exists but no logs yet. Wait a few more minutes and check again.")
            
except Exception as e:
    print(f"❌ Error querying logs: {e}")
    print("\nIf the table doesn't exist, the model wrapper may not have logged yet.")